<a href="https://colab.research.google.com/github/tanveeeD/EmergencyCall_Classifier/blob/main/notebooks/04_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/EmergencyCallProject/dataset/final_dataset.csv")

df.head()


,transcript,label
0,"officer to comix, 6-2-a, where the side drives...",police
1,i would you say this better? what's matter? ca...,medical
2,"i'm a mom with a jammer, can you see? hey, thi...",police
3,"911, what's your margin team? somebody just se...",police
4,beep 6-5-0 for delmore and autoaction and con ...,fire


In [ ]:
label_map = {"fire": 0, "police": 1, "medical": 2}

df["label"] = df["label"].map(label_map)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["transcript"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [ ]:
import torch

class EmergencyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmergencyDataset(train_encodings, train_labels)
test_dataset = EmergencyDataset(test_encodings, test_labels)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.840441
2,No log,0.830756
3,No log,0.849100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=213, training_loss=0.7807323473719924, metrics={'train_runtime': 118.6806, 'train_samples_per_second': 14.257, 'train_steps_per_second': 1.795, 'total_flos': 224138835652608.0, 'train_loss': 0.7807323473719924, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.830759584903717,
 'eval_runtime': 2.1821,
 'eval_samples_per_second': 65.074,
 'eval_steps_per_second': 8.249,
 'epoch': 3.0}

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, y_pred))

              precision    recall  f1-score   support

           0       0.65      0.68      0.67        47
           1       0.75      0.75      0.75        64
           2       0.66      0.61      0.63        31

    accuracy                           0.70       142
   macro avg       0.69      0.68      0.68       142
weighted avg       0.70      0.70      0.70       142



In [ ]:
import torch

text = "Does anyone know CPR?"

device = next(model.parameters()).device

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

reverse_map = {0:"fire", 1:"police", 2:"medical"}

print(reverse_map[pred])

medical


In [ ]:
model.save_pretrained("/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model/tokenizer_config.json',
 '/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model/tokenizer.json')